# Facial Emotion Recognition (EfficientNet-B0) - corrected pipeline

**Replaces `Untitled43.ipynb`.** Same architecture and fine-tuning depth, corrected protocol.

### What changed and why

| # | Original fault | Fix in this notebook |
|---|---|---|
| F1 | The manuscript claims FER2013 + ExpW + supplementary data, but the notebook trains on the Kaggle `emotion-recognition-dataset` **only**. | Dataset provenance recorded explicitly in the results JSON. **Correct the claim in Section 3.1 of the paper.** |
| F2 | Checkpoint saved at `best_val_acc`, then that same number reported. 85.29% is the max of a noisy 83-85% plateau, selected on the data it is reported on. | Checkpoint selected on **validation loss**; a disjoint **test** split is reported. |
| F3 | `torch.randperm` split with no duplicate control. Near-identical images (common when frames come from video) could straddle the split. | Near-duplicates clustered by **perceptual hash**; whole clusters assigned to one split. |
| F4 | Severe overfitting - train accuracy 99.48% while val loss rose from 0.41 to 0.84. | Early stopping on val loss, class weights, and the plateau is no longer cherry-picked. |

**Expect roughly 78-84% on the test split.** Lower than 85.29%, and defensible.

### Splits

| Split | Share | Purpose |
|---|---|---|
| train | 70% | fitting |
| val | 15% | early stopping and checkpoint selection |
| test | 15% | **reported result only** |

Assignment is by duplicate-cluster, stratified by class, and asserted at runtime.

## 1. Data

In [ ]:
!pip install -q kaggle ImageHash
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d sujaykapadnis/emotion-recognition-dataset
!unzip -q -o emotion-recognition-dataset.zip -d emotion_data/

In [ ]:
import os, shutil, json, warnings
warnings.filterwarnings('ignore')

DATA_DIR = 'emotion_data/dataset'
TARGET_EMOTIONS = ['Happy', 'Sad', 'Angry', 'Neutral']

for emotion in sorted(os.listdir(DATA_DIR)):
    folder = os.path.join(DATA_DIR, emotion)
    if os.path.isdir(folder) and emotion not in TARGET_EMOTIONS:
        shutil.rmtree(folder)
        print(f'deleted: {emotion}')
print('\nremaining:', sorted(os.listdir(DATA_DIR)))

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import imagehash
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
PHASH_THRESHOLD = 5          # Hamming distance counted as a near-duplicate
SPLIT = (0.70, 0.15, 0.15)   # train / val / test
BATCH = 64
EPOCHS = 30
PATIENCE = 7

torch.manual_seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device set to: {device}')
print('torch', torch.__version__)

## 2. Deduplicated, cluster-grouped split  *(replaces original Cell 2)*

Two datasets are built over the same folder so training gets augmentation while val and
test do not. `ImageFolder` sorts deterministically, so indices line up between them.

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

full_train_data = datasets.ImageFolder(DATA_DIR, transform=train_transform)
full_eval_data  = datasets.ImageFolder(DATA_DIR, transform=eval_transform)

# NOTE: ImageFolder sorts class folders alphabetically -> Angry, Happy, Neutral, Sad.
# This is NOT the same ordering as the SER 4-class list. Never mix the two.
classes = full_train_data.classes
paths   = [p for p, _ in full_train_data.samples]
labels  = np.array(full_train_data.targets)
n = len(paths)
print(f'Total Images: {n}')
print('Classes (alphabetical):', classes)
print({classes[i]: int((labels == i).sum()) for i in range(len(classes))})

In [ ]:
print('computing perceptual hashes...')
bits = np.zeros((n, 8), dtype=np.uint8)
for i, p in enumerate(paths):
    h = imagehash.phash(Image.open(p).convert('RGB'))
    bits[i] = np.packbits(h.hash.flatten())
    if (i + 1) % 2000 == 0:
        print(f'  {i+1}/{n}')
print('done')

In [ ]:
# ---- cluster near-duplicates: vectorised Hamming distance, chunked ------
POPCOUNT = np.array([bin(v).count('1') for v in range(256)], dtype=np.uint8)
parent = list(range(n))

def find(a):
    while parent[a] != a:
        parent[a] = parent[parent[a]]
        a = parent[a]
    return a

def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra

print('clustering near-duplicates...')
CHUNK = 256
for s in range(0, n, CHUNK):
    e = min(s + CHUNK, n)
    d = POPCOUNT[bits[s:e, None, :] ^ bits[None, :, :]].sum(-1)
    for r in range(e - s):
        i = s + r
        for j in np.nonzero(d[r] <= PHASH_THRESHOLD)[0]:
            if j > i:
                union(i, int(j))

clusters = {}
for i in range(n):
    clusters.setdefault(find(i), []).append(i)
clusters = list(clusters.values())

multi = [cl for cl in clusters if len(cl) > 1]
n_dup = sum(len(cl) for cl in multi)
print(f'  {len(multi)} duplicate clusters covering {n_dup} images '
      f'({100*n_dup/n:.1f}% of the dataset)')
print(f'  largest cluster: {max((len(cl) for cl in multi), default=0)} images')

cross = [cl for cl in clusters if len({labels[i] for i in cl}) > 1]
if cross:
    print(f'  dropping {sum(len(cl) for cl in cross)} images in {len(cross)} '
          f'cross-class clusters (the same face labelled two ways = label noise)')
clusters = [cl for cl in clusters if len({labels[i] for i in cl}) == 1]

In [ ]:
# ---- assign whole clusters, stratified by class -------------------------
by_class = {}
for cl in clusters:
    by_class.setdefault(int(labels[cl[0]]), []).append(cl)

split_idx = {'train': [], 'val': [], 'test': []}
for cls, cl_list in by_class.items():
    order = rng.permutation(len(cl_list))
    total = sum(len(cl_list[i]) for i in order)
    t1, t2 = SPLIT[0] * total, (SPLIT[0] + SPLIT[1]) * total
    acc = 0
    for i in order:
        cl = cl_list[i]
        k = 'train' if acc < t1 else ('val' if acc < t2 else 'test')
        split_idx[k].extend(cl)
        acc += len(cl)

# hard guarantee - no duplicate cluster may straddle a boundary
owner = {}
for k, idxs in split_idx.items():
    for i in idxs:
        owner.setdefault(find(i), set()).add(k)
assert all(len(v) == 1 for v in owner.values()), 'a duplicate cluster spans splits'
print('verified: no duplicate cluster spans a split boundary')

train_dataset = Subset(full_train_data, split_idx['train'])
val_dataset   = Subset(full_eval_data,  split_idx['val'])
test_dataset  = Subset(full_eval_data,  split_idx['test'])

train_loader = DataLoader(train_dataset, batch_size=BATCH, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH, shuffle=False, num_workers=2)

for k in ('train', 'val', 'test'):
    counts = {classes[cix]: int((labels[split_idx[k]] == cix).sum())
              for cix in range(len(classes))}
    print(f'{k:5s}: {len(split_idx[k]):6d} images  {counts}')

with open('fer_split.json', 'w') as f:
    json.dump({k: [[paths[i], int(labels[i])] for i in v]
               for k, v in split_idx.items()}, f)
print('\nsplit saved to fer_split.json - the quantization study must reuse this test set')

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6), sharey=True)
for a, k in zip(ax, ('train', 'val', 'test')):
    counts = [int((labels[split_idx[k]] == cix).sum()) for cix in range(len(classes))]
    sns.barplot(x=classes, y=counts, ax=a, color='#4d4d4d')
    a.set_title(f'{k}  (n={len(split_idx[k])})'); a.tick_params(axis='x', rotation=45)
ax[0].set_ylabel('images')
plt.suptitle('Class distribution per split')
plt.tight_layout(); plt.show()

## 3. Model  *(architecture and fine-tuning depth unchanged)*

In [ ]:
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
for p in model.parameters():
    p.requires_grad = False
for p in model.features[-4:].parameters():
    p.requires_grad = True
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(classes))
model = model.to(device)

ytr = labels[split_idx['train']]
cw = compute_class_weight('balanced', classes=np.arange(len(classes)), y=ytr)
print('class weights:', {classes[i]: round(float(w), 3) for i, w in enumerate(cw)})

criterion = nn.CrossEntropyLoss(weight=torch.tensor(cw, dtype=torch.float32).to(device))
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                              lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
                                                       factor=0.5, patience=3)
print('EfficientNet-B0 ready for', len(classes), 'classes')

## 4. Training

**The key change is one line:** selection is `if val_loss < best_val_loss`, not
`if val_acc > best_val_acc`. Accuracy over a noisy plateau is what produced 85.29%.
`test_loader` is not referenced anywhere in this section.

In [ ]:
@torch.no_grad()
def evaluate(loader):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        loss_sum += criterion(out, y).item() * y.size(0)
        correct += (out.argmax(1) == y).sum().item()
        total += y.size(0)
    return loss_sum / total, 100.0 * correct / total


history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_loss, best_epoch, patience = float('inf'), -1, 0

for epoch in range(EPOCHS):
    model.train()
    run_loss, correct, total = 0.0, 0, 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        run_loss += loss.item() * y.size(0)
        correct += (out.argmax(1) == y).sum().item()
        total += y.size(0)

    trl, tra = run_loss / total, 100.0 * correct / total
    vll, vla = evaluate(val_loader)
    scheduler.step(vll)

    history['train_loss'].append(trl); history['train_acc'].append(tra)
    history['val_loss'].append(vll);   history['val_acc'].append(vla)

    star = ''
    if vll < best_val_loss - 1e-4:                 # <-- selection on LOSS
        best_val_loss, best_epoch, patience = vll, epoch, 0
        torch.save(model.state_dict(), 'jugantarFER_corrected.pth')
        star = '   <- saved'
    else:
        patience += 1

    print(f'Epoch [{epoch+1:2d}/{EPOCHS}] '
          f'Train {trl:.4f}/{tra:5.2f}% | Val {vll:.4f}/{vla:5.2f}%{star}')

    if patience >= PATIENCE:
        print(f'early stopping at epoch {epoch+1}')
        break

print(f'\nbest epoch (min val loss): {best_epoch+1}')
print(f'val accuracy there: {history["val_acc"][best_epoch]:.2f}%'
      '   <- SELECTION metric, not the reported result')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(history['train_acc'], label='train')
ax[0].plot(history['val_acc'], label='val')
ax[0].set_title('Accuracy (%)'); ax[0].set_xlabel('epoch'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(history['train_loss'], label='train')
ax[1].plot(history['val_loss'], label='val')
ax[1].axvline(best_epoch, ls='--', c='k', lw=1, label='selected')
ax[1].set_title('Loss'); ax[1].set_xlabel('epoch'); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 5. Evaluation  *(replaces original Cell 6)*

Both splits are reported so the selection-optimism gap is explicit. **Publish the TEST row.**

In [ ]:
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, f1_score, balanced_accuracy_score)

model.load_state_dict(torch.load('jugantarFER_corrected.pth'))
model.eval().to(device)


@torch.no_grad()
def predict(loader):
    P, Y = [], []
    for x, y in loader:
        P.append(torch.softmax(model(x.to(device)), 1).cpu().numpy())
        Y.append(y.numpy())
    return np.concatenate(P), np.concatenate(Y)


scores = {}
for name, loader in (('VAL (selection set)', val_loader), ('TEST (held out)', test_loader)):
    prob, y = predict(loader)
    pred = prob.argmax(1)
    acc = accuracy_score(y, pred)
    scores[name] = acc

    print('=' * 64)
    print(name)
    print('=' * 64)
    print(classification_report(y, pred, target_names=classes, digits=3, zero_division=0))
    print(f'accuracy          {acc*100:.2f}%')
    print(f'balanced accuracy {balanced_accuracy_score(y, pred)*100:.2f}%')
    macro = f1_score(y, pred, average='macro', zero_division=0)
    print(f'macro F1          {macro:.3f}')

    boot = np.empty(2000)
    for b in range(2000):
        i = rng.integers(0, len(y), len(y))
        boot[b] = (y[i] == pred[i]).mean()
    lo, hi = np.quantile(boot, [0.025, 0.975])
    print(f'95% CI            [{lo*100:.2f}, {hi*100:.2f}]\n')

    if name.startswith('TEST'):
        test_acc, test_macro, test_ci = acc, macro, (lo, hi)
        cm = confusion_matrix(y, pred)
        plt.figure(figsize=(7.5, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Greys', cbar=False,
                    linewidths=1, linecolor='black',
                    xticklabels=classes, yticklabels=classes, annot_kws={'size': 13})
        plt.title(f'FER, held-out test split\naccuracy {acc*100:.2f}%', fontsize=13)
        plt.ylabel('True'); plt.xlabel('Predicted')
        plt.tight_layout(); plt.show()

gap = (scores['VAL (selection set)'] - scores['TEST (held out)']) * 100
print(f'>> selection optimism: val is {gap:+.2f} points above test.')
print('>> Publish the TEST number. The original 85.29% was a val figure.')

In [ ]:
results = {
    'dataset': {
        'source': 'kaggle sujaykapadnis/emotion-recognition-dataset',
        'IMPORTANT': ('The manuscript claims FER2013 + ExpW + supplementary data. '
                      'That does not describe this model. Correct Section 3.1.'),
        'classes_alphabetical': classes,
    },
    'protocol': {
        'split': 'pHash-cluster-grouped, class-stratified, 70/15/15',
        'phash_threshold': PHASH_THRESHOLD,
        'selection_criterion': 'minimum validation loss',
        'limitation': ('pHash removes near-duplicate IMAGES; different photographs of the '
                       'same person may still span splits. State this in the paper.'),
    },
    'n_images': {k: len(v) for k, v in split_idx.items()},
    'duplicate_clusters': len(multi),
    'images_in_duplicate_clusters': int(n_dup),
    'best_epoch': best_epoch + 1,
    'val_accuracy_selection_metric': float(scores['VAL (selection set)']),
    'test_accuracy_REPORT_THIS': float(test_acc),
    'test_macro_f1': float(test_macro),
    'test_ci95': [float(test_ci[0]), float(test_ci[1])],
    'test_confusion_matrix': cm.tolist(),
    'selection_optimism_points': float(gap),
    'originally_reported': {'value': 85.29,
                            'why_invalid': ('maximum validation accuracy over 25 epochs on '
                                            'a random split with no duplicate control, '
                                            'reported from the same set used to select the '
                                            'checkpoint')},
}
with open('fer_corrected_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print(json.dumps(results, indent=2))

## 6. Optional: ONNX export for the quantization study

In [ ]:
dummy = torch.randn(1, 3, 224, 224, device=device)
torch.onnx.export(model, dummy, 'jugantarFER_corrected.onnx',
                  input_names=['input'], output_names=['logits'],
                  dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}},
                  opset_version=13)

import onnxruntime as ort
x = np.random.randn(4, 3, 224, 224).astype(np.float32)
with torch.no_grad():
    ref = model(torch.from_numpy(x).to(device)).cpu().numpy()
got = ort.InferenceSession('jugantarFER_corrected.onnx').run(None, {'input': x})[0]
diff = float(np.abs(ref - got).max())
print(f'ONNX max |logit diff| vs PyTorch: {diff:.3e}')
print('OK' if diff < 1e-3 else 'WARNING: export diverges - investigate before quantizing')

## 7. What to put in the paper

- **Section 3.1 and the abstract:** replace 85.29% with `test_accuracy_REPORT_THIS`.
- **Section 3.1:** correct the training-data claim. This model was trained on the Kaggle
  emotion-recognition dataset only, not FER2013 + ExpW + supplementary data.
- **Methods:** state the split protocol - "near-duplicates clustered by perceptual hash
  (Hamming distance <= 5) with whole clusters assigned to a single split; checkpoint
  selected on validation loss; test split disjoint and untouched during training".
- **Limitations:** pHash removes duplicate images, not repeat photographs of the same
  person. Say so rather than overclaiming.
- `selection_optimism_points` quantifies how much reporting the val figure inflated the
  original number. Quote it in the correction note.
- Keep `jugantarFER_corrected.pth`, `jugantarFER_corrected.onnx` and `fer_split.json` -
  the quantization study must reuse this exact test set.